# PokouAI — Export Gemma 4 fine-tune to LiteRT-LM (`.litertlm`)

Takes the merged HF checkpoint from `03_finetune_cocoa_unsloth.ipynb` (or `05_merge_from_checkpoint.ipynb` if recovering from a timed-out training run) and produces the artefact the mobile app sideloads or downloads:

- `model.litertlm` (~2.5 GB for E2B int4) — LiteRT-LM bundle. Loaded on iOS / Android by `react-native-litert-lm@0.3.7`'s `createLLM().loadModel(path)`. Supports multimodal (`sendMessageWithImage`).

**What changed (May 1, 2026)**: Google's `litert-torch export_hf` command makes Gemma 4 fine-tune → `.litertlm` a single supported step. The earlier dance through `ai-edge-torch` + `litert-lm-builder` is obsolete; the new tool handles re-authoring, quantization, packaging, and the chat-template metadata in one shot.

- Docs: https://ai.google.dev/edge/litert-lm/models/gemma-4 (section "Deploy from Safetensors")
- `--externalize_embedder` is the flag that handles Gemma 4's Per-Layer Embeddings correctly — this is what made the GGUF route fail in llama.cpp (the PLE tables weren't carried into the forward graph).

**Why not GGUF**: llama.cpp Gemma 4 multimodal support is incomplete (issue #22243 — PLE forward graph). A converted GGUF comes out under-sized because the embedder tables are dropped. Skip GGUF for Gemma 4 until that's fixed upstream.

**Prereq**: `/kaggle/working/cocoa_v1_<variant>_merged/` must exist (from notebook 03's merge step, or from notebook 05 if recovering from a checkpoint). Must contain merged safetensors — LoRA-only adapters won't work; run `model.merge_and_unload()` first.

**Disk**: merged HF dir (~5 GB E2B / ~10 GB E4B) + the exported `.litertlm` (~2.5 GB E2B int4 / ~3.6 GB E4B int4). Fits in /kaggle/working (20 GB) with headroom.

**Final validation must happen on a physical device** — iOS Simulator x86 builds and x86_64 Android emulators don't run the AI Edge native SDKs. iPhone 15 Pro / Pixel 8+ recommended for usable t/s on E2B.

## 0 — Parameters

In [ ]:
VARIANT = 'e2b'  # 'e2b' or 'e4b' — must match what notebook 03 / 05 produced

import os
MERGED_DIR    = f'/kaggle/working/cocoa_v1_{VARIANT}_merged'
OUTPUT_DIR    = f'/kaggle/working/cocoa_v1_{VARIANT}_litertlm'
LITERTLM_PATH = f'{OUTPUT_DIR}/model.litertlm'  # name produced by `litert-torch export_hf`

# Chat template override: ensures the runtime gets Gemma 4's correct jinja template
# embedded in the .litertlm metadata. Points at Google's official packaged base for
# the same variant; `export_hf` copies the chat_template from that repo's tokenizer_config.
CHAT_TEMPLATE_REPO = f'litert-community/gemma-4-{VARIANT.upper()}-it-litert-lm'

assert os.path.isdir(MERGED_DIR), (
    f'missing {MERGED_DIR} — re-run notebook 03, or 05 if recovering from a checkpoint'
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'variant       : {VARIANT}')
print(f'merged source : {MERGED_DIR}')
print(f'output dir    : {OUTPUT_DIR}')
print(f'chat template : {CHAT_TEMPLATE_REPO}')
!ls -la {MERGED_DIR} | head -15
!df -h /kaggle/working

## 1 — Install `litert-torch-nightly` and `litert-lm`

Plain pip install puts the CLI binaries on `/usr/local/bin` which the `!` subshell sees. `uv tool install` puts them in `~/.local/bin` which is NOT on the subshell's PATH — avoid that here even though uv is otherwise nice for isolation.

In [ ]:
# 1. Fix protobuf first. litert-torch-nightly's gencode is built with protobuf
#    6.x; Kaggle's image ships protobuf 5.29.5. Mismatch → VersionError at
#    `litert-torch` import time, no .litertlm gets written. Must pin >= 6.31.1.
!pip install -q --no-cache-dir -U "protobuf>=6.31.1"

# 2. torchao must be upgraded: litert-torch-nightly imports
#    torchao.quantization.pt2e which only exists in torchao >= 0.6.
#    Kaggle ships an older torchao (pulled in by unsloth during training).
!pip install -q --no-cache-dir -U torchao

# 3. transformers from github main: Kaggle's wheel doesn't yet have the
#    `gemma4` model_type registration. Without this, `AutoConfig.from_pretrained`
#    raises KeyError('gemma4') inside `litert-torch export_hf` at load time.
!pip install -q --no-cache-dir -U "git+https://github.com/huggingface/transformers.git"

# 4. Now the LiteRT tools. Plain pip drops binaries on /usr/local/bin which
#    the ! subshell sees (uv tool install puts them on ~/.local/bin which it
#    does not — avoid uv here even though it's nicer for isolation).
!pip install -q --no-cache-dir litert-torch-nightly litert-lm

# 5. Verify everything resolves BEFORE the export cell burns 15 min on a
#    silently-busted env. All checks run in subprocesses (fresh Python)
#    so they reflect what `!litert-torch export_hf` will actually see —
#    not what's cached in the kernel's sys.modules (which may still have
#    the old protobuf if grpc/tensorflow/etc loaded it at kernel startup).
!python -c "import google.protobuf; print(f'protobuf            : {google.protobuf.__version__}  (need >= 6.31.1)')"
!python -c "import torchao.quantization.pt2e; print('torchao.quantization.pt2e ok')"
!python -c "import transformers; print(f'transformers        : {transformers.__version__}')"
!python -c "from transformers import AutoConfig; c = AutoConfig.from_pretrained('{MERGED_DIR}'); print(f'config class        : {type(c).__name__}  (need Gemma4Config)')"
!python -c "from litert_torch.cli import main; print('litert_torch import ok')"
!which litert-torch && which litert-lm

## 2 — Export merged HF → `.litertlm`

Single command. `export_hf` reads the merged safetensors, re-authors them for the LiteRT-LM runtime, quantizes (int4 by default for E2B/E4B), and packages everything (weights + tokenizer + chat template + sampler config) into one `.litertlm` file.

**Flags**:

- `--externalize_embedder` — pulls the Per-Layer Embedder tables out as a separate component inside the bundle. Required for Gemma 4 E2B / E4B; without this the bundle is missing the PLE weights and inference produces garbage. This is the exact thing that made the GGUF path fail.
- `--jinja_chat_template_override` — points at the official Gemma 4 LiteRT-LM repo so the chat template (the `<start_of_turn>` / `<end_of_turn>` jinja) is copied verbatim from a known-good source. Our merged dir's `tokenizer_config.json` has the same template, but using the override avoids any drift from training-time edits.

**Vision/audio**: `export_hf` reads the multimodal towers directly from the safetensors when present. No separate vision step needed — the bundle handles `sendMessageWithImage` on-device.

Expected runtime: ~5–15 min on Kaggle CPU; faster on GPU instance. Output size: ~2.5 GB (E2B int4) or ~3.6 GB (E4B int4).

In [ ]:
import time, os, shutil, glob

# Wipe stale outputs from previous attempts.
if os.path.isdir(OUTPUT_DIR):
    print(f'removing stale {OUTPUT_DIR}')
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Export to /tmp (~1.3 TB on Kaggle VMs) instead of /kaggle/working (20 GB
# user quota). The MLIR work_dir lives inside output_dir as a tmpXXXX/
# subdir and holds 5–10 GB of intermediate IR — won't fit in /kaggle/working.
#
# Cannot use --work_dir to redirect: litert-torch 0.x has a kwarg collision
# ("got multiple values for keyword argument 'work_dir'") because output_dir
# already derives work_dir internally. So we point output_dir itself at /tmp
# and shutil.move the final non-temp artifacts back to /kaggle/working.
TMP_OUTPUT_DIR = f'/tmp/cocoa_v1_{VARIANT}_litertlm'
if os.path.isdir(TMP_OUTPUT_DIR):
    shutil.rmtree(TMP_OUTPUT_DIR)
os.makedirs(TMP_OUTPUT_DIR, exist_ok=True)

print(f'free /kaggle/working : {shutil.disk_usage("/kaggle/working").free/1e9:.1f} GB')
print(f'free /tmp            : {shutil.disk_usage("/tmp").free/1e9:.1f} GB')

t0 = time.time()
# Memory + disk budget on Kaggle is tight. To fit:
# - --prefill_lengths=[128] + --cache_length=1024 → smaller prefill graph
#   constants. 1024 ctx is enough for the cocoa diagnosis prompt
#   (~600 tokens system + image + reply).
# - --quantization_recipe=weight_only_wi4_afp32 → int4 weights instead of
#   default int8. Halves model bytes in the MLIR module, getting it under
#   the 2 GB single-file LLVM limit that crashed the int8 attempt.
# - --externalize_embedder → PLE tables in a separate component, required
#   for Gemma 4 E2B/E4B.
!litert-torch export_hf \
    --model={MERGED_DIR} \
    --output_dir={TMP_OUTPUT_DIR} \
    --externalize_embedder \
    --prefill_lengths='[128]' \
    --cache_length=1024 \
    --quantization_recipe=weight_only_wi4_afp32 \
    --jinja_chat_template_override={CHAT_TEMPLATE_REPO}
elapsed = time.time() - t0
print(f'\nexport took {elapsed:.0f}s')

# Move all non-temp artifacts back to /kaggle/working so they persist
# into Kaggle Versions. Skip the tmpXXXX MLIR scratch dir.
# Cross-filesystem shutil.move falls back to copy+delete (tmpfs → ext4);
# on ~2.5 GB that's ~3 seconds, negligible vs the export.
for path in glob.glob(f'{TMP_OUTPUT_DIR}/*'):
    name = os.path.basename(path)
    if name.startswith('tmp'):  # MLIR work_dir — discard
        continue
    shutil.move(path, os.path.join(OUTPUT_DIR, name))
shutil.rmtree(TMP_OUTPUT_DIR, ignore_errors=True)

assert os.path.isfile(LITERTLM_PATH), f'export did not produce {LITERTLM_PATH}'
sz_gb = os.path.getsize(LITERTLM_PATH) / 1e9
print(f'\n✓ {LITERTLM_PATH} ({sz_gb:.2f} GB)')
!ls -lh {OUTPUT_DIR}

## 3 — Smoke-test the `.litertlm` with `litert-lm run`

Cheap to do in Kaggle, expensive to debug on the phone. `litert-lm run` is the same engine `react-native-litert-lm` ships on-device — if a text prompt produces coherent output here, the bundle will load on the phone too.

This is text-only because Kaggle has no easy way to feed it an image. Vision is validated on the physical device.

In [ ]:
# Fine-tune-aware prompt: a domain-specific cocoa pathology question.
# Base Gemma 4 will give generic answers ("brown spots can have many causes...").
# A correctly-carried-through cocoa fine-tune should name Phytophthora palmivora,
# describe the rot pattern, and reference humidity/rainfall as a risk factor —
# whatever your training data emphasized.
#
# If output is generic, the fine-tune didn't carry through the export (re-check
# that you exported from the *merged* dir, not the LoRA-only adapter dir).
!litert-lm run {LITERTLM_PATH} --prompt="Décris les symptômes de la pourriture brune sur cacaoyer."

## 4 — Get the file to the phone

Two paths, depending on whether you want auto-download on first launch or a manual sideload during dev.

**Path A — Sideload (dev / immediate)**

1. Download `model.litertlm` from Kaggle's *Output* panel (right side of notebook).
2. Rename to `cocoa_v1_e2b.litertlm` (matches the constant in `LiteRTService.ts`).
3. Finder → iPhone → Files → PokouAI → drop the file in.
4. App's `LiteRTService.isModelReady()` will see the file on disk and prefer it over the upstream Gemma 3n.

**Path B — HuggingFace upload (production / first-launch auto-download)**

Push to a public or private HF repo; the smoke screen can download from any HTTPS URL via `llm.downloadModel(url, fileName, onProgress)`. Caches under iOS `Library/Caches/litert_models/` and Android internal storage — first-launch downloads once, all subsequent inference is offline.

The submission "offline" requirement is satisfied by this pattern: one download on first launch, then everything runs locally.

In [ ]:
# Optional: push to HF Hub. Uncomment + paste a write-scope HF token.
# from huggingface_hub import HfApi, login
# login(token='hf_...')  # write scope
# REPO = f'<your-username>/pokouai-cocoa-{VARIANT}-litertlm'
# api = HfApi()
# api.create_repo(REPO, exist_ok=True, private=True)
# api.upload_file(
#     path_or_fileobj=LITERTLM_PATH,
#     path_in_repo='model.litertlm',
#     repo_id=REPO,
# )
# print(f'pushed → https://huggingface.co/{REPO}/resolve/main/model.litertlm')

## 5 — Notes

**Why `--externalize_embedder` matters**: Gemma 4 E2B/E4B use Per-Layer Embeddings (PLE) — large embedding tables consulted per-decoder-layer. The default `export_hf` would inline them; for the E2B/E4B variants the tables are large enough that inlining either fails or produces a bundle that won't load on phone memory. Externalizing them keeps the bundle on-disk-paged at inference time. This is the same root cause as the GGUF size issue you saw — llama.cpp's converter silently drops the PLE tables.

**Why `--jinja_chat_template_override`**: belt-and-braces. The merged HF dir's `tokenizer_config.json` has the chat template the fine-tune was trained against, but `export_hf` uses the override to copy from a known-good Google-published source, avoiding any chance of a malformed template breaking inference on-device.

**Multimodal is included automatically**: if your fine-tune was trained multimodal (notebook 03 trained on cocoa pod images), `export_hf` picks up the vision tower from the merged safetensors. No separate flag, no separate `.tflite`. The vision projector that ended up in your old `cocoa_v1_e2b-mmproj.gguf` is irrelevant here — the LiteRT-LM pipeline pulls vision directly from the HF safetensors.

**Validation must happen on a physical device**: `litert-lm run` here proves the bundle is structurally sound and the text path works. Vision / audio / actual device performance only validate on an iPhone or Pixel. The app's *Settings → 🧪 LiteRT smoke test* screen is the right test bed — it logs t/s, TTFT, RSS, and runs a real `sendMessageWithImage`.

**If `export_hf` errors**:

- *Missing weights*: you likely have LoRA adapters, not merged weights. Re-run notebook 05's merge cell (the one that calls `model.merge_and_unload()`).
- *Out of memory*: drop to E2B if you were trying E4B, or run on a Kaggle GPU instance for extra RAM via offload.
- *Unknown architecture*: `litert-torch-nightly` version may have drifted from the Gemma 4 release. Force-upgrade: `pip install -U --force-reinstall --no-cache-dir litert-torch-nightly`.
- *`command not found`*: the `!` subshell can't find the binary. The install cell uses plain `pip` to drop it on `/usr/local/bin` for this reason; if you switched to `uv tool install`, add `os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']` before the export cell.

**v1.1 follow-up**: once on-device validation is clean, port the `--jinja_chat_template_override` arg into notebook 03 as a comment-only reminder so the next training cycle knows what the export expects, and consider pushing the `.litertlm` to a public HF repo so the app's first-launch download story is documented in the submission.